# 3. Train and exam

Trains the LoRA on the training set from notebook 2, merges it, converts to q8, and runs the blind before/after exam.

Settings: Accelerator = GPU T4 x2, Internet = On, and the output of notebook 2 attached as input. Use **Save Version > Save & Run All (Commit)** so it keeps running with the browser closed.

In [ ]:
EPOCHS = 3    # run 1 used 2
HF_REPO = ""  # optional, e.g. "your-name/qwen25-7b-harness-manual-lora"; needs a notebook secret HF_TOKEN

In [ ]:
REPO_URL = "https://github.com/Naveenasri02/qwen25-lora-book-pilot.git"

import glob, json, os, shutil, subprocess, sys, zipfile
os.environ["HF_HOME"] = "/tmp/hf"          # big downloads go to scratch space, not the 20 GB output folder
REPO = "/tmp/repo"
if not os.path.exists(REPO):
    # the project comes from an attached dataset if there is one, otherwise from GitHub
    attached = glob.glob("/kaggle/input/**/pilot/config.py", recursive=True)
    zipped = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO)
    elif zipped:
        zipfile.ZipFile(zipped[0]).extractall(REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
sys.path.insert(0, REPO)
from pilot import judge, ollama_client as oc
from pilot.config import (BASE_OLLAMA, FRESH_QUESTIONS_PATH, JUDGE_OLLAMA, QUESTIONS_PATH, SEED,
                          TUNED_OLLAMA, UNITS_PATH)
OUT = "/kaggle/working"
questions = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
fresh = json.loads(FRESH_QUESTIONS_PATH.read_text(encoding="utf-8"))   # second exam, written for run 2
print(len(questions), "exam questions and", len(fresh), "fresh exam questions loaded")

## Step 1: QLoRA training on one T4

In [ ]:
TRAIN_FILE = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0]   # written by notebook 2
print("training set:", TRAIN_FILE, sum(1 for _ in open(TRAIN_FILE, encoding="utf-8")), "examples")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes"], check=True)
# some notebook images carry an old torchao that peft refuses to load next to; nothing here uses it
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pilot.train", "--train", TRAIN_FILE,
                "--out", f"{OUT}/adapter", "--epochs", str(EPOCHS)],
               cwd=REPO, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"), check=True)
shutil.rmtree(f"{OUT}/adapter/checkpoints", ignore_errors=True)
print(open(f"{OUT}/adapter/training_config.json").read())

## Step 2: merge, convert to q8, load into ollama (the build that gets examined)

In [ ]:
from pilot import export
subprocess.run([sys.executable, "-m", "pilot.export", "merge", "--adapter", f"{OUT}/adapter",
                "--out", "/tmp/merged"], cwd=REPO, env=dict(os.environ, CUDA_VISIBLE_DEVICES=""), check=True)
shutil.rmtree("/tmp/hf", ignore_errors=True)          # make room on disk
export.to_gguf_q8("/tmp/merged", "/tmp/tuned-q8.gguf")
shutil.rmtree("/tmp/merged", ignore_errors=True)
oc.start_server()
oc.pull(BASE_OLLAMA)   # the tuned build reuses the base model's chat template
export.ollama_create("/tmp/tuned-q8.gguf", TUNED_OLLAMA)
os.remove("/tmp/tuned-q8.gguf")

## Step 3: blind exam, before and after

Both models answer both exams (the original 175 questions and the fresh 175) as q8 in ollama with the same system prompt. The answers are pooled and shuffled, and Phi-4 marks each one without knowing which model wrote it.

In [ ]:
oc.pull(JUDGE_OLLAMA)
all_results = {"judge": JUDGE_OLLAMA, "base": BASE_OLLAMA}
for exam_name, exam in (("original", questions), ("fresh", fresh)):
    before = judge.answer_all(oc.chat_many, BASE_OLLAMA, exam)
    after = judge.answer_all(oc.chat_many, TUNED_OLLAMA, exam)
    sheet, key = judge.make_blind_sheet(exam, {"before": before, "after": after}, SEED)
    labels = judge.judge_sheet(oc.chat_many, JUDGE_OLLAMA, sheet)
    results = judge.score(sheet, key, labels, exam)
    all_results[exam_name] = {"results": results, "sheet": sheet, "labels": labels, "key": key}

    print(f"{exam_name + ' exam':28}{'before':>8}{'after':>8}")
    for name, field in [("correct of 100", "correct_of_100"), ("made-up answers of 60", "made_up_of_60"),
                        ("refusals on practical of 15", "refusals_of_15")]:
        print(f"{name:28}{results['before'][field]:>8}{results['after'][field]:>8}")
    for arm in ("before", "after"):
        r = results[arm]
        print(arm, "| set A:", r["set_a"], "| direct:", r["set_a_direct"], "| indirect:", r["set_a_indirect"],
              "| unparsed:", r["unparsed"])
json.dump(all_results, open(f"{OUT}/exam_results.json", "w"), indent=1)

## Step 4: pack the adapter (and upload it to Hugging Face if HF_REPO is set)

In [ ]:
shutil.make_archive(f"{OUT}/adapter", "zip", f"{OUT}/adapter")
if HF_REPO:
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HF_REPO, private=True, exist_ok=True)   # private until you decide to publish
    api.upload_folder(folder_path=f"{OUT}/adapter", repo_id=HF_REPO)
    print("uploaded to", HF_REPO)
print(sorted(os.listdir(OUT)))